# 04 — Deploy SageMaker Endpoint (MAESTRO)

Deploy the GST Entity Matcher as a SageMaker real-time endpoint on MAESTRO,
then attach it to API Gateway.

**Run this on MAESTRO SageMaker JupyterLab.**

Prerequisites:
- Run `02_run_indexing.ipynb` first (FAISS index must exist in S3)
- You need a SageMaker Project created in MAESTRO

After deployment:
1. Attach the endpoint to API Gateway via MAESTRO UI (Domains Pane → Domain Details → API Gateway)
2. Set `SAGEMAKER_ENDPOINT_URL` and `SAGEMAKER_API_KEY` on Airbase

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import time
from datetime import datetime

import boto3
import sagemaker

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# SageMaker SDK client (uses MAESTRO config from lcc-init-script)
sagemaker_client = boto3.client('sagemaker', region_name='ap-southeast-1')

# Pre-populated vars from MAESTRO
ASSUME_ROLE = sagemaker.get_execution_role()
S3_BUCKET = sagemaker.session.Session().default_bucket()

# Domain ID
with open('/opt/ml/metadata/resource-metadata.json') as f:
    metadata = json.load(f)
    DOMAIN_ID = metadata.get('DomainId')

# ======================================================================
# YOUR SAGEMAKER PROJECT — update these to match your MAESTRO project
# ======================================================================
SAGEMAKER_PROJECT_ID = 'p-88q9jscqrh0h'
SAGEMAKER_PROJECT_NAME = 'gst-registrants'

# MAESTRO platform vars — DO NOT CHANGE
ENV = 'prod'
AWS_REGION = 'ap-southeast-1'
AWS_ACCOUNT_ID = '819382625810'
SAGEMAKER_PIPELINE_ROLE_ARN = f'{ASSUME_ROLE}'
SAGEMAKER_PROJECT_NAME_ID = f'{SAGEMAKER_PROJECT_NAME}-{SAGEMAKER_PROJECT_ID}'

# Endpoint settings
INSTANCE_TYPE = 'ml.m5.large'  # ~400MB FAISS index fits in 8GB RAM
INSTANCE_COUNT = 1
SAMPLING_PERCENTAGE = '100'
DATA_CAPTURE_UPLOAD_PATH = f's3://{S3_BUCKET}/capture/'
ENABLE_DATA_CAPTURE = 'true'

# Git commit for resource naming
CI_COMMIT_SHORT_SHA = !git rev-parse --short HEAD
CI_COMMIT_SHORT_SHA = CI_COMMIT_SHORT_SHA[0] if CI_COMMIT_SHORT_SHA else 'latest'

print(f'Role: {ASSUME_ROLE}')
print(f'S3 Bucket: {S3_BUCKET}')
print(f'Domain ID: {DOMAIN_ID}')
print(f'Project: {SAGEMAKER_PROJECT_NAME} ({SAGEMAKER_PROJECT_ID})')

## 2. Package model and upload to S3

Create `model.tar.gz` with the inference code and upload to S3.

In [ ]:
os.chdir(PROJECT_ROOT)

from endpoint.package_model import package_model

package_model()

from config import S3_BUCKET as GST_BUCKET, S3_EMBEDDINGS_PREFIX
MODEL_DATA_URL = f's3://{GST_BUCKET}/{S3_EMBEDDINGS_PREFIX}model.tar.gz'
print(f'Model artifact: {MODEL_DATA_URL}')

## 3. Get domain metadata

Retrieve VPC and KMS settings from the MAESTRO domain (required for endpoint deployment).

In [ ]:
print('=' * 80)
print('GETTING DOMAIN METADATA')
print('=' * 80)

response = sagemaker_client.describe_domain(DomainId=DOMAIN_ID)

SUBNET_ID_1 = response['SubnetIds'][0]
SUBNET_ID_2 = response['SubnetIds'][1]
SECURITY_GROUP_ID = response['DefaultUserSettings']['SecurityGroups'][0]

print(f'Subnet 1: {SUBNET_ID_1}')
print(f'Subnet 2: {SUBNET_ID_2}')
print(f'Security Group: {SECURITY_GROUP_ID}')

# KMS key for encryption
kms_client = boto3.client('kms', region_name=AWS_REGION)
kms_key_alias = f'alias/cmk-agml-{ENV}-domain-{DOMAIN_ID}'
kms_response = kms_client.describe_key(KeyId=kms_key_alias)
KMS_KEY_ARN = kms_response['KeyMetadata']['Arn']

print(f'KMS Key ARN: {KMS_KEY_ARN}')
print('\n✓ Domain metadata retrieved successfully')

## 4. Get container image

Retrieve the PyTorch inference container image URI.
Update the framework/version if your MAESTRO environment uses a different image.

In [ ]:
image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch',
    region=AWS_REGION,
    version='2.0',
    py_version='py310',
    instance_type=INSTANCE_TYPE,
    image_scope='inference',
)
print(f'Image URI: {image_uri}')

## 5. Create SageMaker Model

Register the model with SageMaker using our container image + model.tar.gz.

Note: Unlike the MLOps pipeline demo (which uses a `ModelPackageName` from the model registry),
we create the model directly from the container image and model data, since our "model" is
a FAISS index + inference code — not a trained ML model.

In [ ]:
print('=' * 80)
print('CREATING SAGEMAKER MODEL')
print('=' * 80)

timestamp = int(datetime.now().timestamp())
MODEL_NAME = f'gst-entity-matcher-{CI_COMMIT_SHORT_SHA}-{timestamp}'
CONFIG_NAME = f'gst-entity-matcher-config-{CI_COMMIT_SHORT_SHA}-{timestamp}'
ENDPOINT_NAME = f'gst-entity-matcher-{CI_COMMIT_SHORT_SHA}'

print(f'\nResource names:')
print(f'  Model Name: {MODEL_NAME}')
print(f'  Config Name: {CONFIG_NAME}')
print(f'  Endpoint Name: {ENDPOINT_NAME}')

try:
    response = sagemaker_client.create_model(
        ModelName=MODEL_NAME,
        PrimaryContainer={
            'Image': image_uri,
            'ModelDataUrl': MODEL_DATA_URL,
            'Environment': {
                'OPENAI_API_KEY': os.getenv('OPENAI_API_KEY', ''),
                'OPENAI_API_BASE': os.getenv('OPENAI_API_BASE', ''),
            },
        },
        ExecutionRoleArn=ASSUME_ROLE,
        EnableNetworkIsolation=True,
        VpcConfig={
            'SecurityGroupIds': [SECURITY_GROUP_ID],
            'Subnets': [SUBNET_ID_1, SUBNET_ID_2],
        },
        Tags=[
            {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
            {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
        ],
    )
    print(f'\n✓ Model created: {response["ModelArn"]}')
except sagemaker_client.exceptions.ResourceInUse:
    print(f'⚠ Model {MODEL_NAME} already exists, continuing...')

## 6. Create Endpoint Configuration

In [ ]:
print('=' * 80)
print('CREATING ENDPOINT CONFIGURATION')
print('=' * 80)

try:
    response = sagemaker_client.create_endpoint_config(
        EndpointConfigName=CONFIG_NAME,
        ProductionVariants=[
            {
                'VariantName': 'AllTraffic',
                'ModelName': MODEL_NAME,
                'InstanceType': INSTANCE_TYPE,
                'InitialInstanceCount': INSTANCE_COUNT,
                'InitialVariantWeight': 1.0,
            }
        ],
        DataCaptureConfig={
            'EnableCapture': ENABLE_DATA_CAPTURE == 'true',
            'InitialSamplingPercentage': int(SAMPLING_PERCENTAGE),
            'DestinationS3Uri': DATA_CAPTURE_UPLOAD_PATH,
            'CaptureOptions': [
                {'CaptureMode': 'Input'},
                {'CaptureMode': 'Output'},
            ],
            'CaptureContentTypeHeader': {
                'JsonContentTypes': ['application/json'],
            },
            'KmsKeyId': KMS_KEY_ARN,
        },
        KmsKeyId=KMS_KEY_ARN,
        Tags=[
            {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
            {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
            {'Key': 'sagemaker:domain-arn', 'Value': f'arn:aws:sagemaker:{AWS_REGION}:{AWS_ACCOUNT_ID}:domain/{DOMAIN_ID}'},
        ],
    )
    print(f'\n✓ Endpoint config created: {response["EndpointConfigArn"]}')
except sagemaker_client.exceptions.ResourceInUse:
    print(f'⚠ Endpoint config {CONFIG_NAME} already exists, continuing...')

## 7. Create or Update Endpoint

In [ ]:
print('=' * 80)
print('CREATING/UPDATING SAGEMAKER ENDPOINT')
print('=' * 80)

# Check if endpoint already exists
endpoint_exists = False
try:
    endpoint_response = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_exists = True
    print(f'\n✓ Endpoint exists with status: {endpoint_response["EndpointStatus"]}')
except sagemaker_client.exceptions.ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code in ['AccessDeniedException', 'ValidationException']:
        print(f'\nEndpoint does not exist (got {error_code})')
        endpoint_exists = False
    else:
        raise

# Update or Create
if endpoint_exists:
    print(f'\nUpdating endpoint: {ENDPOINT_NAME}')
    response = sagemaker_client.update_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=CONFIG_NAME,
    )
    print('✓ Endpoint update initiated')
else:
    print(f'\nCreating new endpoint: {ENDPOINT_NAME}')
    response = sagemaker_client.create_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=CONFIG_NAME,
        Tags=[
            {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
            {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
            {'Key': 'sagemaker:domain-arn', 'Value': f'arn:aws:sagemaker:{AWS_REGION}:{AWS_ACCOUNT_ID}:domain/{DOMAIN_ID}'},
        ],
    )
    print('✓ Endpoint creation initiated')

print(f'\nEndpoint: {ENDPOINT_NAME}')
print(f'Config: {CONFIG_NAME}')
print(f'Model: {MODEL_NAME}')

## 8. Wait for Endpoint to be InService

This takes a few minutes. The cell polls every 30 seconds.

In [ ]:
print('=' * 80)
print('WAITING FOR ENDPOINT TO BE IN SERVICE')
print('=' * 80)

TIMEOUT = 1800  # 30 minutes
ELAPSED = 0

print(f'\nMonitoring endpoint: {ENDPOINT_NAME}')
print(f'Timeout: {TIMEOUT}s (30 minutes)')
print('-' * 80)

while ELAPSED < TIMEOUT:
    try:
        endpoint_response = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        CURRENT_STATUS = endpoint_response['EndpointStatus']

        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f'[{ts}] Status: {CURRENT_STATUS} | Elapsed: {ELAPSED}s')

        if CURRENT_STATUS == 'InService':
            print('-' * 80)
            print('✓ Endpoint is now in service!')
            break
        elif CURRENT_STATUS == 'Failed':
            print('-' * 80)
            print('✗ Endpoint creation/update failed!')
            if 'FailureReason' in endpoint_response:
                print(f'Failure reason: {endpoint_response["FailureReason"]}')
            break

        time.sleep(30)
        ELAPSED += 30

    except Exception as e:
        print(f'Error checking status: {e}')
        break

if ELAPSED >= TIMEOUT:
    print('-' * 80)
    print('⏱️  Timeout — endpoint may still be creating in the background')

## 9. Test the endpoint

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name=AWS_REGION)

test_payload = json.dumps({'entity_names': ['DBS BANK', 'SINGAPORE AIRLINES']})

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Accept='application/json',
    Body=test_payload,
)

results = json.loads(response['Body'].read().decode('utf-8'))
print(json.dumps(results, indent=2))

## 10. Next steps

Now that the endpoint is InService:

1. **Attach to API Gateway:** Go to MAESTRO UI → Domains Pane → Domain Details → API Gateway → attach this endpoint
2. **Get the API URL and API Key** from the API Gateway page
3. **Set env vars on Airbase:**
   - `SAGEMAKER_ENDPOINT_URL` = the API Gateway URL
   - `SAGEMAKER_API_KEY` = the API key
4. **Deploy the Streamlit app** to Airbase

In [ ]:
print('=' * 60)
print('DEPLOYMENT COMPLETE')
print('=' * 60)
print()
print(f'Endpoint Name: {ENDPOINT_NAME}')
print(f'Model Name:    {MODEL_NAME}')
print(f'Config Name:   {CONFIG_NAME}')
print()
print('Next steps:')
print('1. Attach endpoint to API Gateway in MAESTRO UI')
print('2. Get API URL + API Key from API Gateway page')
print('3. Set SAGEMAKER_ENDPOINT_URL and SAGEMAKER_API_KEY on Airbase')
print('4. Deploy Streamlit app to Airbase')

## Cleanup (optional)

Uncomment to delete the endpoint and stop incurring costs.

In [ ]:
# Uncomment to delete
# sagemaker_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
# sagemaker_client.delete_endpoint_config(EndpointConfigName=CONFIG_NAME)
# sagemaker_client.delete_model(ModelName=MODEL_NAME)
# print(f'Deleted endpoint, config, and model.')